<a href="https://colab.research.google.com/github/shouryadengre12-bot/Fundamentals-of-AI-and-ML-Evaluated-Course-Project/blob/main/BYOP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install "transformers<5" torch spacy
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
import spacy
from transformers import pipeline

class LectureAssistant:
    def __init__(self):
        print("Loading AI models... (This might take a minute on the first run)")
        # Load Hugging Face Summarization Pipeline (BART model)
        self.summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

        # Load spaCy for Named Entity Recognition (NER)
        self.nlp = spacy.load("en_core_web_sm")
        print("Models loaded successfully!\n")

    def summarize_lecture(self, text):
        """Generates a concise summary from a long lecture transcript."""
        print("Generating summary...")
        # We set max and min lengths for the output summary
        # If the text is very long, in a real app you would chunk it.
        # For this prototype, we assume a reasonable paragraph/page.
        input_length = len(text.split())
        max_len = min(130, int(input_length * 0.6))

        summary = self.summarizer(text, max_length=max_len, min_length=30, do_sample=False)
        return summary[0]['summary_text']

    def generate_flashcards(self, text):
        """Creates fill-in-the-blank flashcards by blanking out key entities."""
        print("Generating flashcards...")
        doc = self.nlp(text)
        flashcards = []

        # Process the text sentence by sentence
        for sent in doc.sents:
            # Look for important entities like Dates, People, Organizations, Concepts
            entities = [ent.text for ent in sent.ents]

            # If no entities found, fallback to important noun phrases
            if not entities:
                entities = [chunk.text for chunk in sent.noun_chunks if len(chunk.text.split()) > 1]

            # If we found an important keyword, create a flashcard
            if entities:
                # Pick the first important concept to blank out
                answer = entities[0]
                # Create the question by replacing the answer with blanks
                question = sent.text.replace(answer, "________")

                flashcards.append({
                    "Question": question.strip(),
                    "Answer": answer.strip()
                })

        return flashcards

# ==========================================
# Main Execution
# ==========================================
if __name__ == "__main__":
    # Example lecture text (You can replace this with `open('lecture.txt', 'r').read()`)
    sample_lecture = """
    Machine learning is a subfield of artificial intelligence. The primary goal of machine learning
    is to understand the structure of data and fit that data into models that can be understood
    and utilized by people. In 1959, Arthur Samuel coined the term machine learning, defining it as
    a field of study that gives computers the ability to learn without being explicitly programmed.
    Today, neural networks are heavily used in modern machine learning tasks like image recognition.
    """

    # Initialize our assistant
    assistant = LectureAssistant()

    # 1. Get Summary
    summary = assistant.summarize_lecture(sample_lecture)
    print("\n--- LECTURE SUMMARY ---")
    print(summary)

    # 2. Get Flashcards (Based on the summary for high-yield studying)
    print("\n--- STUDY FLASHCARDS ---")
    flashcards = assistant.generate_flashcards(summary)

    for i, card in enumerate(flashcards, 1):
        print(f"\nCard {i}:")
        print(f"Q: {card['Question']}")
        print(f"A: {card['Answer']}")

Loading AI models... (This might take a minute on the first run)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


Models loaded successfully!

Generating summary...

--- LECTURE SUMMARY ---
Machine learning is a subfield of artificial intelligence. In 1959, Arthur Samuel coined the term machine learning. Today, neural networks are heavily used in modern machine learning tasks like image recognition.

--- STUDY FLASHCARDS ---
Generating flashcards...

Card 1:
Q: ________ is a subfield of artificial intelligence.
A: Machine learning

Card 2:
Q: In ________, Arthur Samuel coined the term machine learning.
A: 1959

Card 3:
Q: ________, neural networks are heavily used in modern machine learning tasks like image recognition.
A: Today
